In [ ]:
#| hide
from my_blog.core import *

# my-blog

> A personal blog built with FastHTML, MonsterUI, and nbdev. Features HTMX-powered SPA-like navigation, tag-based filtering, Obsidian markdown authoring, and role-based access control via fasthtml-auth.

## Overview

This repository contains a blog developed using [FastHTML](https://fastht.ml), [MonsterUI](https://github.com/answerdotai/monsterui), and [nbdev](https://nbdev.fast.ai). The key source notebooks are:

- **`05_blog_v5.ipynb`** — exports to `my_blog/core_v5.py`. Contains the main application factory (`create_app`), all public routes (home, blog listing, individual posts), database setup, markdown rendering with custom extensions, and UI components (navbar, footer, layout).
- **`06_admin.ipynb`** — exports to `my_blog/admin_v1.py`. Contains admin-only routes for editing, saving, deleting, and downloading posts.

Earlier notebooks (`00_core`, `01_blog_v2`, etc.) have been kept as a development history showing how the codebase evolved.

The blog makes extensive use of HTMX to replicate a Single Page Application experience — partial page swaps mean only the `#main-content` div is updated on navigation, keeping the navbar and footer in place without a full reload.

Posts are written in [Obsidian](https://obsidian.md), uploaded via the blog's upload route, and stored in a SQLite database (managed via [fastlite](https://github.com/answerdotai/fastlite)). Posts can be assigned tags for filtering. Custom markdown extensions support Obsidian-style image sizing/alignment and embedded Strava routes.

Online editing of post content is possible for administrators. Currently tags and frontmatter cannot be changed in the online editor — download the `.md` file, edit locally, and re-upload to change those.

---

## Post Creation

Posts are authored in Obsidian and uploaded to the blog. Each post file should include YAML frontmatter:

```yaml
---
title: My Post Title
excerpt: A short summary shown in listings.
tags:
  - cycling
  - technology
date: 2024-01-15
published: true
private: false
---

Post content here...
```

- **`title`** — used to generate the URL slug (lowercased, spaces → hyphens, truncated to 60 chars). E.g. "A Visit to Spain" → `/blog/a-visit-to-spain`.
- **`excerpt`** — shown in post listing cards.
- **`tags`** — list of tag strings; used for the tag filter on the blog page.
- **`published`** — set to `true` to make the post visible.
- **`private`** — set to `true` to restrict visibility to logged-in users only.

If a post with the same slug already exists it will be **updated** rather than duplicated.

---

## Image Display

Images use Obsidian's wiki-link syntax with optional size and alignment:

```
![[image_filename.jpg]]              # default display
![[image_filename.jpg|250]]          # 250px wide
![[image_filename.jpg|250|right]]    # 250px, float right
![[image_filename.jpg|400|center]]   # 400px, centred
![[image_filename.jpg|300|left]]     # 300px, float left
```

Images for a post are uploaded into a folder named after the post's slug:
`static/image/post_images/{slug}/image_filename.jpg`

After markdown rendering, the `process_obsidian_images` function injects inline CSS to apply the requested size and alignment.

---

## Strava Embeddings

Strava activity maps can be embedded using a custom tag:

```
{{strava:17555511761}}
```

Where the number is the Strava activity (ride) ID. After rendering, `process_strava_embeddings` replaces these tags with the appropriate Strava embed markup.

---

## Routes and Access

### Public routes (no login required)

| Route | Description |
|---|---|
| `GET /` | Homepage — intro section + latest 4 posts |
| `GET /about` | About page |
| `GET /blog` | Full post listing with tag filter |
| `GET /blog/{slug}` | Individual post (private posts redirect to login) |

### Authentication routes (via fasthtml-auth)

| Route | Description |
|---|---|
| `GET/POST /auth/login` | Login form |
| `GET /auth/logout` | Logout and redirect to `/` |
| `GET/POST /auth/register` | Registration (if `ALLOW_REGISTRATION=true`) |
| `GET/POST /auth/profile` | Logged-in user profile management |
| `GET /auth/google/login` | Google OAuth login (if configured) |
| `GET /auth/google/callback` | Google OAuth callback |

### Admin routes (require `admin` role)

| Route | Description |
|---|---|
| `GET /admin/edit/{slug}` | Side-by-side markdown editor with live preview |
| `POST /admin/save/{slug}` | Save edited post content |
| `POST /admin/delete/{slug}` | Delete a post |
| `GET /admin/download/{slug}` | Download post as `.md` file with frontmatter |
| `GET /auth/admin` | User management dashboard |
| `GET /auth/admin/users` | List / search / filter users |
| `GET/POST /auth/admin/users/create` | Create a new user |
| `GET/POST /auth/admin/users/edit?id={id}` | Edit user details and role |
| `GET/POST /auth/admin/users/delete?id={id}` | Delete a user |

---

## Configuration Setup

Create a `.env` file in the project root with the following variables:

```ini
# --- Required ---
SECRET_KEY=your-secret-key-here          # Used for session signing

# --- Admin account (applied on startup) ---
ADMIN_USERNAME=admin
ADMIN_PASSWORD=your-admin-password
ADMIN_EMAIL=you@example.com

# --- Paths (optional, sensible defaults provided) ---
STATIC_DIR=/path/to/static               # Default: <project_root>/static
DATA_DIR=/path/to/data                   # Default: my_blog/data
USERS_DB_NAME=users.db
POSTS_DB_NAME=posts.db

# --- Registration ---
ALLOW_REGISTRATION=false                 # Set true to allow public sign-up

# --- Google OAuth (optional) ---
GOOGLE_CLIENT_ID=your-google-client-id
GOOGLE_CLIENT_SECRET=your-google-client-secret
OAUTH_REDIRECT_URL=https://yourdomain.com/auth/google/callback

# --- Server ---
HOST=0.0.0.0
PORT=5000
DEBUG=false
```

---

## User Management

User management is handled by [fasthtml-auth](https://github.com/fromlittleacorns/fasthtml-auth). Three roles are supported:

| Role | Access |
|---|---|
| `user` | Can view private posts when logged in |
| `manager` | Manager-level access (superset of user) |
| `admin` | Full access including post editing and user management |

The admin dashboard at `/auth/admin` allows creating, editing, and deleting users, with search/filter and pagination. The default admin credentials (`admin` / `admin123`) are overwritten on startup with the values from `ADMIN_USERNAME` / `ADMIN_PASSWORD` in `.env`.

Google OAuth can be enabled so users sign in with their Google account. By default OAuth logins auto-create accounts; set `oauth_create_users=False` in the auth config (and `ALLOW_REGISTRATION=false`) to restrict sign-in to admin-pre-created accounts only.

See the [fasthtml-auth README](https://github.com/fromlittleacorns/fasthtml-auth) for full details.

This file becomes the README and the index page of the nbdev documentation site.

## Developer Guide

### Running Locally

Clone the repo and install in development mode:

```sh
git clone https://github.com/fromLittleAcorns/my-blog.git
cd my-blog
pip install -e .
```

Copy `.env.example` to `.env` and fill in your values (see Configuration Setup above). Then, after making changes to any notebook, export to Python with:

```sh
nbdev_prepare
```

### Running the Blog Server

For production, start the blog with:

```sh
python blog.py
```

For development, use `dev.py` which enables auto-reload on code changes:

```sh
python dev.py
```

The blog will be available at `http://localhost:5000`. The admin interface is at `/auth/admin`.

See the sections above for full usage documentation.